# Lab 02 — Data Preprocessing & Vocabulary Construction

In [195]:
lines = [
    "1 Mary moved to the bathroom.",
    "2 John went to the hallway.",
    "3 Where is Mary? \tbathroom\t1",
    "4 Daniel went back to the hallway.",
    "5 Sandra moved to the garden."
]

## Challenge 1 — Tokenization & Vocabulary Building

### Tasks
- [ ] Remove the sentence ID from a raw bAbI line.
- [ ] Convert text to lowercase.
- [ ] Remove trailing punctuation (`.` or `?`).
- [ ] Split text into tokens.
- [ ] Create a reusable `tokenize(text)` function.
- [ ] Tokenize multiple sentences.
- [ ] Collect unique words using `set()`.
- [ ] Sort vocabulary using `sorted()`.
- [ ] Assign word IDs using `enumerate()`.
- [ ] Create a `word2id` dictionary.
- [ ] Display vocabulary size and word IDs.

### 1.1 — Inspect and tokenize a raw line

In [196]:
line = lines[0] ## input
line = line.lower().split(" " , 1)[1]
line = line.rstrip(".").split() ## remove the period at the end of the sentence
print(line)

['mary', 'moved', 'to', 'the', 'bathroom']


### 1.2 — Define example sentences and tokenizer

In [197]:
sentences = [
    "Mary went to the garden.",
    "John went to the kitchen."
]

In [198]:
def tokenize(text):
    tokens = text.lower().rstrip(".?").split()
    return tokens

print([tokenize(sentence) for sentence in sentences])

[['mary', 'went', 'to', 'the', 'garden'], ['john', 'went', 'to', 'the', 'kitchen']]


### 1.3 — Build the vocabulary and word IDs

In [199]:
## defining a set to collect unique tokens
vocab = {token for sentence in sentences for token in tokenize(sentence)}


In [200]:
word2id = dict()

for i, v in enumerate(sorted(vocab)):
    word2id[v] = i

In [201]:
print(word2id)
print(len(word2id))


{'garden': 0, 'john': 1, 'kitchen': 2, 'mary': 3, 'the': 4, 'to': 5, 'went': 6}
7


## Challenge 2 — Text Encoding 

### Tasks
- [ ] Create a function encode(text).
- [ ] Tokenize the input using tokenize().
- [ ] Retrieve each token's ID from word2id.
- [ ] Return a list of token IDs.

In [202]:
sentence = "Mary went to the garden."  ## input 
def encode(text):
    T = list()
    tokens = tokenize(text)
    for token in tokens:
        if token in word2id:
            T.append(word2id[token])

    return T

encode(sentence)

[3, 6, 5, 4, 0]

## Challenge 3 — Answer Label Encoding

### Tasks
- [ ] Define a correct answer (`answer`).
- [ ] Normalize the answer using lowercase.
- [ ] Retrieve the answer ID from `word2id`.
- [ ] Store the result in `answer_id`.
- [ ] Display the answer and its ID.

In [203]:
answer = "Garden"

if answer.lower() in word2id:
    answer_id = word2id[answer.lower()]
    print(answer_id)

0


## Challenge 4 — Padding & Masks

### Tasks
- [ ] Understand why sequences need padding.
- [ ] Define a special `<PAD>` token.
- [ ] Find the maximum sequence length.
- [ ] Pad shorter sequences to equal lengths.
- [ ] Create attention masks (1 = real token, 0 = padding).

### 4.1 — Example sequences

In [204]:
sequences = [
    [3, 6, 5, 4, 0],
    [1, 6, 5, 2]
]

### 4.2 — Padding and masks

In [205]:
max = 0
for seq in sequences:
    if len(seq) > max:
        max = len(seq)


for seq in sequences:
    pad = max - len(seq)
    seq.extend([7] * pad)

print(sequences)

masks =  []
for seq in sequences:
    mask = []
    for n in seq:
        if n == 7:
            mask.append(0)
        else:
            mask.append(1)

    masks.append(mask)

print(masks)


[[3, 6, 5, 4, 0], [1, 6, 5, 2, 7]]
[[1, 1, 1, 1, 1], [1, 1, 1, 1, 0]]


## Challenge 5 — PyTorch Tensors

### Tasks
- [ ] Convert padded sequences into PyTorch tensors.
- [ ] Convert masks into PyTorch tensors.
- [ ] Use the `torch.long` data type.
- [ ] Display tensor shapes using `.shape`.

In [206]:
import torch

tsequences = torch.tensor(sequences, dtype=torch.long)
print(tsequences)
print(tsequences.shape)


tmasks = torch.tensor(masks, dtype=torch.long)
print(tmasks)
print(tmasks.shape)


tensor([[3, 6, 5, 4, 0],
        [1, 6, 5, 2, 7]])
torch.Size([2, 5])
tensor([[1, 1, 1, 1, 1],
        [1, 1, 1, 1, 0]])
torch.Size([2, 5])


## Challenge 6 — PyTorch Dataset & DataLoader

### Tasks
- [ ] Understand the difference between Dataset and DataLoader.
- [ ] Create a TensorDataset from sequences and masks.
- [ ] Create a DataLoader with batch_size=1.
- [ ] Iterate through the batches.
- [ ] Display each batch and its shape.

In [207]:
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(tsequences , tmasks)

loader = DataLoader(
    dataset,
    batch_size=1
)

for Xbatch, maskbatch in loader:
    print("Sequences:", Xbatch.shape)
    print("Masks:", maskbatch.shape)


Sequences: torch.Size([1, 5])
Masks: torch.Size([1, 5])
Sequences: torch.Size([1, 5])
Masks: torch.Size([1, 5])


## Challenge 7 — Final Tensor Shapes

### Tasks
- [ ] Display the number of samples in the Dataset.
- [ ] Display the number of batches in the DataLoader.
- [ ] Retrieve one batch using `next(iter(loader))`.
- [ ] Display the shape of the sequences and masks.
- [ ] Identify batch size and sequence length.

### Inspect dataset and batch dimensions

In [208]:
print(len(dataset))

print(len(loader))

Xbatch, maskbatch = next(iter(loader))

print(Xbatch.shape)
print( maskbatch.shape)

print(Xbatch.shape[0])
print(Xbatch.shape[1])

2
2
torch.Size([1, 5])
torch.Size([1, 5])
1
5
